In [6]:
import datetime
import logging
import pathlib

import polars as pl
from memory_profiler import profile
from polars import Schema
from polars.datatypes import (
    Boolean,
    Float64,
    Int64,
    List,
    String,
    Struct,
)

ROOT_DIR_PATH = pathlib.Path(".").resolve().parent
DATA_DIR_PATH = ROOT_DIR_PATH / "data"
FAKE_GAMING_DATA_DIR = DATA_DIR_PATH / "fake_gaming_data"

## Define Schema

In [7]:
schema = Schema(
    {
        "team_id": String,
        "name": String,
        "created_date": String,
        "ranking": Int64,
        "total_winnings": Int64,
        "members": List(
            Struct(
                {
                    "player_id": String,
                    "username": String,
                    "account_details": Struct(
                        {
                            "email": String,
                            "registration_date": String,
                            "premium_status": Boolean,
                            "country": String,
                            "language": String,
                        }
                    ),
                    "stats": Struct(
                        {
                            "level": Int64,
                            "experience": Int64,
                            "total_matches": Int64,
                            "win_rate": Float64,
                            "playtime_hours": Int64,
                            "achievements_completed": Int64,
                        }
                    ),
                    "inventory": Struct(
                        {
                            "currency": Struct({"premium": Int64, "standard": Int64}),
                            "items": List(
                                Struct(
                                    {
                                        "item_id": String,
                                        "name": String,
                                        "type": String,
                                        "rarity": String,
                                        "level_requirement": Int64,
                                        "stats": Struct(
                                            {
                                                "attack": Int64,
                                                "defense": Int64,
                                                "magic": Int64,
                                                "speed": Int64,
                                            }
                                        ),
                                    }
                                )
                            ),
                        }
                    ),
                    "achievements": List(
                        Struct(
                            {
                                "id": String,
                                "name": String,
                                "difficulty": String,
                                "completion_rate": Float64,
                                "points": Int64,
                                "date": String,
                            }
                        )
                    ),
                    "recent_matches": List(
                        Struct(
                            {
                                "match_id": String,
                                "game_mode": String,
                                "map": String,
                                "duration_minutes": Int64,
                                "date": String,
                                "stats": Struct(
                                    {
                                        "kills": Int64,
                                        "deaths": Int64,
                                        "assists": Int64,
                                        "damage_dealt": Int64,
                                        "healing_done": Int64,
                                        "accuracy": Float64,
                                        "headshot_percentage": Float64,
                                        "objectives_completed": Int64,
                                    }
                                ),
                                "rewards": Struct(
                                    {
                                        "experience": Int64,
                                        "currency": Int64,
                                        "items_dropped": List(
                                            Struct(
                                                {
                                                    "item_id": String,
                                                    "name": String,
                                                    "rarity": String,
                                                    "value": Int64,
                                                }
                                            )
                                        ),
                                    }
                                ),
                            }
                        )
                    ),
                }
            )
        ),
        "tournament_history": List(
            Struct(
                {
                    "tournament_id": String,
                    "name": String,
                    "placement": Int64,
                    "prize_money": Int64,
                    "matches_played": Int64,
                }
            )
        ),
    }
)

## Read LazyFrame 
(No execution triggered, data is loaded lazily aka. dag generation, like in spark)

In [8]:
lf: pl.LazyFrame = pl.scan_ndjson(
    FAKE_GAMING_DATA_DIR / "data.json",
    schema=schema,
)

In [9]:
lf.explain(optimized=True)

'NDJson SCAN [/Users/jakubpluta/Repositories/priv/polars-internals/data/fake_gaming_data/data.json]\nPROJECT */7 COLUMNS'

In [10]:
sample = lf.limit(5).collect(
    streaming=True
)  # read only 5 records. LazyFrame materialized into DataFrame

In [11]:
sample.head(1)

shape: (1, 7)
┌──────────────┬──────────────┬──────────────┬─────────┬──────────────┬──────────────┬─────────────┐
│ team_id      ┆ name         ┆ created_date ┆ ranking ┆ total_winnin ┆ members      ┆ tournament_ │
│ ---          ┆ ---          ┆ ---          ┆ ---     ┆ gs           ┆ ---          ┆ history     │
│ str          ┆ str          ┆ str          ┆ i64     ┆ ---          ┆ list[struct[ ┆ ---         │
│              ┆              ┆              ┆         ┆ i64          ┆ 7]]          ┆ list[struct │
│              ┆              ┆              ┆         ┆              ┆              ┆ [5]]        │
╞══════════════╪══════════════╪══════════════╪═════════╪══════════════╪══════════════╪═════════════╡
│ 3d11adbe-36d ┆ Squad        ┆ 2024-11-22   ┆ 493     ┆ 498804       ┆ [{"f16ba9c0- ┆ [{"33b25f51 │
│ 0-4af4-9ac1- ┆ Johnson Inc  ┆              ┆         ┆              ┆ f9ea-4079-b2 ┆ -088d-4014- │
│ 194ef7…      ┆              ┆              ┆         ┆              ┆ 06-a8e…      ┆ 8e23-06f…   │
└──────────────┴──────────────┴──────────────┴─────────┴──────────────┴──────────────┴─────────────┘

In [12]:
sample.schema

Schema([('team_id', String),
        ('name', String),
        ('created_date', String),
        ('ranking', Int64),
        ('total_winnings', Int64),
        ('members',
         List(Struct({'player_id': String, 'username': String, 'account_details': Struct({'email': String, 'registration_date': String, 'premium_status': Boolean, 'country': String, 'language': String}), 'stats': Struct({'level': Int64, 'experience': Int64, 'total_matches': Int64, 'win_rate': Float64, 'playtime_hours': Int64, 'achievements_completed': Int64}), 'inventory': Struct({'currency': Struct({'premium': Int64, 'standard': Int64}), 'items': List(Struct({'item_id': String, 'name': String, 'type': String, 'rarity': String, 'level_requirement': Int64, 'stats': Struct({'attack': Int64, 'defense': Int64, 'magic': Int64, 'speed': Int64})}))}), 'achievements': List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})), 'recent_matches': List(Struct(

In [13]:
sample.dtypes

[String,
 String,
 String,
 Int64,
 Int64,
 List(Struct({'player_id': String, 'username': String, 'account_details': Struct({'email': String, 'registration_date': String, 'premium_status': Boolean, 'country': String, 'language': String}), 'stats': Struct({'level': Int64, 'experience': Int64, 'total_matches': Int64, 'win_rate': Float64, 'playtime_hours': Int64, 'achievements_completed': Int64}), 'inventory': Struct({'currency': Struct({'premium': Int64, 'standard': Int64}), 'items': List(Struct({'item_id': String, 'name': String, 'type': String, 'rarity': String, 'level_requirement': Int64, 'stats': Struct({'attack': Int64, 'defense': Int64, 'magic': Int64, 'speed': Int64})}))}), 'achievements': List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})), 'recent_matches': List(Struct({'match_id': String, 'game_mode': String, 'map': String, 'duration_minutes': Int64, 'date': String, 'stats': Struct({'kills': Int64, 'dea

In [14]:
df = lf.collect()  # load whole data into memory

In [15]:
df.shape  # 14701 records, 7 columns

(14701, 7)

In [16]:
df.head(1)

shape: (1, 7)
┌──────────────┬──────────────┬──────────────┬─────────┬──────────────┬──────────────┬─────────────┐
│ team_id      ┆ name         ┆ created_date ┆ ranking ┆ total_winnin ┆ members      ┆ tournament_ │
│ ---          ┆ ---          ┆ ---          ┆ ---     ┆ gs           ┆ ---          ┆ history     │
│ str          ┆ str          ┆ str          ┆ i64     ┆ ---          ┆ list[struct[ ┆ ---         │
│              ┆              ┆              ┆         ┆ i64          ┆ 7]]          ┆ list[struct │
│              ┆              ┆              ┆         ┆              ┆              ┆ [5]]        │
╞══════════════╪══════════════╪══════════════╪═════════╪══════════════╪══════════════╪═════════════╡
│ 3d11adbe-36d ┆ Squad        ┆ 2024-11-22   ┆ 493     ┆ 498804       ┆ [{"f16ba9c0- ┆ [{"33b25f51 │
│ 0-4af4-9ac1- ┆ Johnson Inc  ┆              ┆         ┆              ┆ f9ea-4079-b2 ┆ -088d-4014- │
│ 194ef7…      ┆              ┆              ┆         ┆              ┆ 06-a8e…      ┆ 8e23-06f…   │
└──────────────┴──────────────┴──────────────┴─────────┴──────────────┴──────────────┴─────────────┘

In [17]:
### members, tournament_history  is a list of structs, we can explode them into multi rows

In [18]:
df_members_exploded = df.explode(pl.col("members"))

In [19]:
df_members_exploded.head()

team_id,name,created_date,ranking,total_winnings,members,tournament_history
str,str,str,i64,i64,struct[7],list[struct[5]]
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""Squad Johnson Inc""","""2024-11-22""",493,498804,"{""f16ba9c0-f9ea-4079-b206-a8e0b7737eaa"",""marvinthomas"",{""kevin49@example.com"",""2025-01-02"",true,""Gibraltar"",""ja""},{91,159523,313,50.35,1173,32},{{5067,85179},[{""90f6348c-3f55-46cd-8de4-66c3d1fc0ac9"",""Heavy Armor"",""Accessory"",""Epic"",87,{16,2,90,46}}, {""bf43fee5-9024-4742-bfc9-d9920efe82c8"",""Dragon Sword"",""Accessory"",""Legendary"",10,{72,29,20,74}}, … {""3f715010-385a-449b-bfc5-be92c6255517"",""Mage Staff"",""Accessory"",""Rare"",79,{94,38,54,21}}]},[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}],[{""7f06e144-0d2c-439f-9c98-7a0354f26ef3"",""Ranked"",""Desert Temple"",16,""2024-11-16"",{6,2,15,44202,2655,62.51,28.01,2},{515,231,[{""9e234564-ea7c-490f-993c-6242f47adf23"",""Epic Shield"",""Common"",4508}]}}, {""6643172c-a3b8-47af-9534-37daff187d9d"",""Ranked"",""Arctic Base"",40,""2024-12-31"",{8,1,7,24940,9011,91.92,26.08,6},{412,413,[{""b163b423-1574-4799-874b-f60c5943be55"",""Epic Shield"",""Rare"",1887}, {""d31d543d-d2dc-465f-83d2-607b50e7ff04"",""Mythic Ring"",""Rare"",6528}, {""41fbb686-06e4-47f8-a760-007c2f360551"",""Epic Shield"",""Mythic"",7217}]}}, … {""a12c789c-3099-47b7-9c26-1d4f8ec4cf2d"",""Ranked"",""Arctic Base"",37,""2024-11-14"",{14,9,11,7126,4369,41.58,13.58,0},{851,251,[{""8120762a-1450-4555-a628-c50b7f28dc68"",""Mythic Ring"",""Epic"",8433}, {""67a7cfa1-d0fd-47c2-b53e-a10612d39a3c"",""Mythic Ring"",""Legendary"",826}, {""dc003003-c6e8-4ee0-ada1-fbf03c0138b1"",""Legendary Armor"",""Rare"",9823}]}}]}","[{""33b25f51-088d-4014-8e23-06fd74a6c8ae"",""Pro Series 7"",2,48192,9}, {""b3c18160-022a-4d1b-9299-5d608013971b"",""Champion Series 1"",13,98075,5}, … {""7b48d01e-4255-4061-87a5-163040e8cf0a"",""Pro Series 2"",15,46796,6}]"
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""Squad Johnson Inc""","""2024-11-22""",493,498804,"{""f3c264e6-6b6c-4542-a133-380b531a43ac"",""moniquelowery"",{""joel28@example.net"",""2024-12-05"",true,""Romania"",""en""},{7,13223,609,53.76,4778,67},{{7673,83614},[{""dbd4d2ea-02df-4e19-ac5e-f8d2ad816551"",""Ancient Relic"",""Accessory"",""Rare"",86,{86,58,7,58}}, {""57629505-1959-4551-b397-0c68aeee914a"",""Ancient Relic"",""Armor"",""Common"",94,{15,54,42,54}}, … {""5aa395da-39f1-42ab-a300-c5685d39452d"",""Heavy Armor"",""Consumable"",""Common"",81,{26,47,16,58}}]},[{""5f98e39f-41af-45f3-b433-21cb0c9167db"",""Ultimate Boss Slayer"",""Hard"",29.01,695,""2024-11-09""}, {""3abc0c56-7864-4126-8518-0491712495fa"",""Master of Combat"",""Easy"",67.5,559,""2025-01-17""}, … {""2f805d28-6047-4b84-8da2-f34450b36008"",""Master of Combat"",""Hard"",48.06,562,""2024-11-07""}],[{""41a5c3b3-373d-4487-ab48-35b10eda462a"",""Ranked"",""Jungle Ruins"",31,""2024-11-05"",{18,4,12,1563,1370,44.1,14.12,6},{379,466,[{""1cf5b644-f4d3-4190-8a17-4dcb99c861e3"",""Rare Sword"",""Epic"",4072}, {""a1a5e373-abcc-4338-83e0-c21943dad942"",""Epic Shield"",""Rare"",8673}, {""35d281b5-6161-4e91-b510-78ad5389c861"",""Rare Sword"",""Rare"",2280}]}}, {""6f015e8a-9a22-46dd-851f-9daa306f809c"",""Casual"",""Underground City"",38,""2024-11-05"",{9,18,11,38965,5,52.12,22.1,9},{653,181,[{""dd5d91f9-395a-40c4-acbc-abf49afe3417"",""Legendary Armor"",""Common"",247}, {""354e010a-fb47-462a-ae03-ca993544b62a"",""Mythic Ring"",""Legendary"",1917}, {""1667ebe5-cee5-4fd7-8180-6b5a40cf3f32"",""Rare Sword"",""Common"",6888}]}}, … {""c63577b6-deb9-4265-ae2a-ecbad4b024fb"",""Casual"",""Space Station"",38,""2024-12-11"",{12,14,7,43558,6730,63.8,14.89,0},{310,206,[{""9d375b61-55d7-40cb-96c8-7af158c93c8b"",""Rare Sword"",""Rare"",3578}]}}]}","[{""33b25f51-088d-4014-8e23-06fd74a6c8

In [20]:
df_members_exploded.shape  # we exploded from 14701 -> 110613 records

(110613, 7)

In [21]:
### We can also spread (unnest) struct into multiple columns

In [22]:
df_members_exploded_spread = df_members_exploded.unnest("members")

In [23]:
df_members_exploded_spread.head(1)

team_id,name,created_date,ranking,total_winnings,player_id,username,account_details,stats,inventory,achievements,recent_matches,tournament_history
str,str,str,i64,i64,str,str,struct[5],struct[6],struct[2],list[struct[6]],list[struct[7]],list[struct[5]]
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""Squad Johnson Inc""","""2024-11-22""",493,498804,"""f16ba9c0-f9ea-4079-b206-a8e0b7…","""marvinthomas""","{""kevin49@example.com"",""2025-01-02"",true,""Gibraltar"",""ja""}","{91,159523,313,50.35,1173,32}","{{5067,85179},[{""90f6348c-3f55-46cd-8de4-66c3d1fc0ac9"",""Heavy Armor"",""Accessory"",""Epic"",87,{16,2,90,46}}, {""bf43fee5-9024-4742-bfc9-d9920efe82c8"",""Dragon Sword"",""Accessory"",""Legendary"",10,{72,29,20,74}}, … {""3f715010-385a-449b-bfc5-be92c6255517"",""Mage Staff"",""Accessory"",""Rare"",79,{94,38,54,21}}]}","[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}]","[{""7f06e144-0d2c-439f-9c98-7a0354f26ef3"",""Ranked"",""Desert Temple"",16,""2024-11-16"",{6,2,15,44202,2655,62.51,28.01,2},{515,231,[{""9e234564-ea7c-490f-993c-6242f47adf23"",""Epic Shield"",""Common"",4508}]}}, {""6643172c-a3b8-47af-9534-37daff187d9d"",""Ranked"",""Arctic Base"",40,""2024-12-31"",{8,1,7,24940,9011,91.92,26.08,6},{412,413,[{""b163b423-1574-4799-874b-f60c5943be55"",""Epic Shield"",""Rare"",1887}, {""d31d543d-d2dc-465f-83d2-607b50e7ff04"",""Mythic Ring"",""Rare"",6528}, {""41fbb686-06e4-47f8-a760-007c2f360551"",""Epic Shield"",""Mythic"",7217}]}}, … {""a12c789c-3099-47b7-9c26-1d4f8ec4cf2d"",""Ranked"",""Arctic Base"",37,""2024-11-14"",{14,9,11,7126,4369,41.58,13.58,0},{851,251,[{""8120762a-1450-4555-a628-c50b7f28dc68"",""Mythic Ring"",""Epic"",8433}, {""67a7cfa1-d0fd-47c2-b53e-a10612d39a3c"",""Mythic Ring"",""Legendary"",826}, {""dc003003-c6e8-4ee0-ada1-fbf03c0138b1"",""Legendary Armor"",""Rare"",9823}]}}]","[{""33b25f51-088d-4014-8e23-06fd74a6c8ae"",""Pro Series 7"",2,48192,9}, {""b3c18160-022a-4d1b-9299-5d608013971b"",""Champion Series 1"",13,98075,5}, … {""7b48d01e-4255-4061-87a5-163040e8cf0a"",""Pro Series 2"",15,46796,6}]"


In [24]:
df_members_exploded_spread.shape

(110613, 13)

In [25]:
### we can now access elements from structs

In [26]:
df_members_exploded_spread.select("account_details").schema

Schema([('account_details',
         Struct({'email': String, 'registration_date': String, 'premium_status': Boolean, 'country': String, 'language': String}))])

In [27]:
# extract 2 fileds from struct
df_members_exploded_spread.select(
    pl.col("account_details").struct["country"],
    pl.col("account_details").struct["premium_status"],
)

country,premium_status
str,bool
"""Gibraltar""",true
"""Romania""",true
"""Egypt""",true
"""Malta""",false
"""Antarctica (the territory Sout…",true
…,…
"""Mali""",false
"""New Zealand""",true
"""Burundi""",false


In [28]:
# extract filed from list
df_members_exploded_spread.select("achievements").schema

Schema([('achievements',
         List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})))])

In [29]:
df_members_exploded_spread.select("achievements").head(1)

achievements
list[struct[6]]
"[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}]"


In [30]:
df_members_exploded_spread.select(pl.col("achievements").list.get(0)).head(
    1
)  # first element from list

achievements
struct[6]
"{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}"


In [31]:
df_members_exploded_spread.select(
    pl.col("achievements").list.eval(pl.element().struct["points"])
)  # extract field from list of structs

achievements
list[i64]
"[662, 847, … 729]"
"[695, 559, … 562]"
"[339, 179, … 370]"
"[462, 52, … 780]"
"[293, 23, … 218]"
…
"[525, 107, … 871]"
"[513, 863, … 736]"
"[525, 681, … 410]"


In [32]:
df_members_exploded_spread.select(
    pl.col("achievements").list.eval(pl.element().struct["points"]).list.mean()
)  # calculate mean points

achievements
f64
490.875
544.090909
461.222222
442.25
435.833333
…
502.0
578.75
656.5


In [33]:
df_members_exploded_spread.select(pl.col("achievements")).explode(
    pl.col("achievements")
).unnest("achievements")

id,name,difficulty,completion_rate,points,date
str,str,str,f64,i64,str
"""5695d538-f8c2-462e-b90b-269c93…","""Ultimate Boss Slayer""","""Extreme""",45.57,662,"""2024-12-28"""
"""59e568e5-4f56-492d-8e21-100193…","""100% Completion""","""Medium""",24.42,847,"""2025-01-13"""
"""c26f4010-0162-4e96-9497-d0f651…","""Speed Runner""","""Extreme""",56.01,109,"""2025-01-09"""
"""c187c0e0-231f-4ef4-b5c3-2e9216…","""Master of Combat""","""Extreme""",31.29,836,"""2024-11-14"""
"""451377a4-498b-4fbc-8405-82f4d3…","""Speed Runner""","""Hard""",97.69,277,"""2025-01-04"""
…,…,…,…,…,…
"""821baf37-9aee-481f-9b6f-c17456…","""100% Completion""","""Hard""",82.26,755,"""2024-12-19"""
"""d1fd0e35-69b3-4457-857d-5cecc7…","""Speed Runner""","""Medium""",57.41,47,"""2024-12-31"""
"""d409dce4-439c-4e6c-bbfb-3fec15…","""Master of Combat""","""Easy""",94.94,650,"""2024-12-25"""


In [36]:
df_members_exploded_spread.select("team_id", "player_id", "achievements").explode(
    pl.col("achievements")
).unnest("achievements").group_by("team_id", "player_id").agg(
    pl.col("completion_rate").mean().alias("avg_player_completion_rate"),
    pl.col("points").mean().alias("avg_player_points"),
)

team_id,player_id,avg_player_completion_rate,avg_player_points
str,str,f64,f64
"""a59d2a26-9825-4391-a896-a4bfbe…","""ba162c51-31eb-461b-a3f0-cb3405…",46.466,558.1
"""cb7156bf-2ca8-400f-b23e-7a019c…","""580e8f10-f3be-4ec4-8cd4-abd1a2…",52.85,613.333333
"""015c5d16-ddb8-41c0-914c-986bcd…","""b42364ea-2cec-4ddc-8058-e1fa7d…",55.958,295.2
"""78a8d714-e3dc-4014-a03d-4974a9…","""fa2b1651-1c14-4b1e-93cf-ce9397…",54.9625,443.75
"""867996a2-e6ca-4c4b-a4e4-449c6a…","""627e3c7f-ba2b-44d8-9127-e1450f…",30.14625,580.75
…,…,…,…
"""0da33d28-82c1-44d9-8d19-d2573d…","""584ef26a-4f69-45c3-8891-9cf455…",37.384444,605.111111
"""e6ad3da1-8f69-4598-a081-a31886…","""38ae3fb0-ee82-4372-8a7d-83dc46…",48.0,490.428571
"""c8a9785c-1514-4b0f-aa84-0ac56d…","""8cbf0705-4822-484d-a516-934519…",44.053529,456.647059
